# 7 — Despliegue: API, contenedor y dashboard Human-in-the-loop

**Entrada:** `ml_mejor_modelo.json` (notebook 4), `dataset_modelado.csv`, `harmonie_crudo.csv`
**Salida:** `modelo_export/booster.json`, `modelo_produccion.json`, `app.py`, `streamlit_dashboard.py`,
`Dockerfile`, `docker-compose.yml`, `requirements.txt`

Cierra el objetivo 5 del Anexo I —la parte de MLOps que no es *tracking*— y el objetivo 6
(*Human-in-the-loop*). El notebook 7 registró los experimentos en MLflow; este pone el modelo
**a servir** y construye la interfaz de decisión.

| Capa | Pieza | Se ejecuta |
|---|---|---|
| Modelo | `modelo_export/booster.json` | Se genera aquí |
| Servicio | `app.py` (FastAPI) | Proceso continuo, se arranca y prueba desde aquí |
| Contenedor | `Dockerfile`, `docker-compose.yml` | Se generan y validan aquí |
| Interfaz | `streamlit_dashboard.py` | Proceso continuo, se lanza al final |

## Por qué el modelo se exporta en formato nativo de XGBoost

La API **no** carga el modelo desde el Model Registry de MLflow, sino como `Booster` nativo.
Dos motivos concretos:

1. **Portabilidad de rutas.** Con artefactos en disco local, MLflow guarda en su base de datos
   la ruta *absoluta* de la máquina donde se registró. Esa ruta no existe dentro del contenedor.
2. **Compatibilidad de versiones.** El wrapper `XGBRegressor` serializa atributos internos de
   scikit-learn que cambian entre versiones mayores; el formato nativo del `Booster` es estable
   entre versiones por diseño.

Como consecuencia el contenedor no necesita `mlflow` instalado, solo `xgboost`: imagen más
pequeña y arranque más rápido. MLflow sigue siendo la herramienta de *entrenamiento y registro*
(notebook 7), no de *servicio*.

> **Aviso de sincronía.** El modelo desplegado debe corresponder al del capítulo de resultados.
> La sección 1 lo comprueba explícitamente y aborta si no coinciden: servir un modelo distinto
> al documentado invalidaría la trazabilidad del TFM.

In [1]:
import warnings; warnings.filterwarnings('ignore')
from pathlib import Path
import json, os, time, signal, subprocess, sys

import numpy as np
import pandas as pd
import requests
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import TimeSeriesSplit
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

RUTA     = Path('.')
LATENCIA = 4
OBJ      = 'obs_vel'
H_PROD   = 12        # horizonte de operación: punto medio del rango evaluado
UMBRAL   = 11.5      # m/s, umbral de activación de seguridad
SEMILLA  = 42
PUERTO_API = 8000
N_JOBS   = 4         # M5: 4 núcleos de rendimiento (ver notebook 4)

with open(RUTA/'ml_mejor_modelo.json') as f:
    ml_info = json.load(f)

W_PROD = ml_info['ventana_por_horizonte'][str(H_PROD)]
met_prod = next(m for m in ml_info['metricas'] if m['H'] == H_PROD)
HP_GUARDADOS = met_prod.get('params')      # None si el notebook 4 no los guardó

print(f"Modelo a desplegar: {ml_info['ganador']} H={H_PROD}h, ventana={W_PROD}")
print(f"MAE de referencia (notebook 4): {met_prod['MAE']:.4f}")
print(f"Hiperparámetros guardados: {'sí' if HP_GUARDADOS else 'NO (se reoptimiza abajo)'}")

Modelo a desplegar: XGBoost H=12h, ventana=12
MAE de referencia (notebook 4): 1.5207
Hiperparámetros guardados: sí


---
## 1. Reconstrucción del modelo de producción

Las features se construyen con la **misma función** que el notebook 4, para que el modelo
servido reciba exactamente la misma representación con la que se entrenó. Cualquier
divergencia aquí sería un *training-serving skew*: el fallo más habitual —y más difícil de
detectar— al poner un modelo en producción.

Si el notebook 4 guardó los hiperparámetros, se reutilizan tal cual. Si no (versiones
anteriores los descartaban al serializar), se reoptimiza **solo este horizonte**: reconstruir
con valores por defecto daría un modelo ~10% peor que el documentado.

In [2]:
def construir_dataset(H, W):
    """Idéntica al notebook 4: lags por tramo + cíclicas + HARMONIE + tendencia."""
    crudo = pd.read_csv(RUTA/'harmonie_crudo.csv', parse_dates=['valida', 'pasada'])
    crudo['V']   = np.hypot(crudo.WSPE, crudo.WSPN)
    crudo['Dir'] = (np.degrees(np.arctan2(-crudo.WSPE, -crudo.WSPN)) + 360) % 360
    apto = crudo[crudo.lead >= H + LATENCIA].sort_values(['valida', 'pasada'])
    oper = apto.groupby('valida').last()
    harm = pd.DataFrame({'H_vel': oper['V'], 'H_lead': oper['lead'],
                         'H_dir_sin': np.sin(np.radians(oper['Dir'])),
                         'H_dir_cos': np.cos(np.radians(oper['Dir']))})
    harm.index.name = 'ts'

    base = pd.read_csv(RUTA/'dataset_modelado.csv', index_col='ts', parse_dates=['ts'])
    d = (base.drop(columns=[c for c in base.columns if c.startswith('H_')], errors='ignore')
             .join(harm, how='inner'))
    g = d.groupby('tramo')
    LAGS = [f'lag{k}' for k in range(H, H+W)]
    for k in range(H, H+W):
        d[f'lag{k}'] = g[OBJ].shift(k)
    d['lag_media'] = d[LAGS].mean(axis=1)
    d['lag_std']   = d[LAGS].std(axis=1)
    d['lag_tend']  = d[LAGS[0]] - d[LAGS[-1]]
    F = LAGS + ['hora_sin', 'hora_cos', 'dia_sin', 'dia_cos',
                'H_vel', 'H_dir_sin', 'H_dir_cos', 'H_lead',
                'lag_media', 'lag_std', 'lag_tend']
    return d.dropna(subset=F + [OBJ]), F

d_prod, F_PROD = construir_dataset(H_PROD, W_PROD)
n = len(d_prod)
tr, va, te = d_prod.iloc[:int(n*.6)], d_prod.iloc[int(n*.6):int(n*.8)], d_prod.iloc[int(n*.8):]
dev = pd.concat([tr, va])
print(f'{n} filas, {len(F_PROD)} features | dev={len(dev)} test={len(te)}')

if HP_GUARDADOS:
    hp = {k: v for k, v in HP_GUARDADOS.items()
          if k not in ('n_jobs', 'verbosity', 'random_state')}
    print('Hiperparámetros reutilizados del notebook 4.')
else:
    print(f'Reoptimizando H={H_PROD} (60 trials)...')
    def objetivo(t):
        p = dict(max_depth=t.suggest_int('max_depth', 3, 9),
                 learning_rate=t.suggest_float('learning_rate', .005, .3, log=True),
                 subsample=t.suggest_float('subsample', .5, 1.0),
                 colsample_bytree=t.suggest_float('colsample_bytree', .5, 1.0),
                 min_child_weight=t.suggest_int('min_child_weight', 1, 20),
                 gamma=t.suggest_float('gamma', 1e-8, 5, log=True),
                 reg_alpha=t.suggest_float('reg_alpha', 1e-8, 10, log=True),
                 reg_lambda=t.suggest_float('reg_lambda', 1e-8, 10, log=True))
        maes = []
        for i_tr, i_va in TimeSeriesSplit(n_splits=3, gap=H_PROD).split(dev):
            m = XGBRegressor(**p, n_estimators=2000, early_stopping_rounds=30,
                             objective='reg:absoluteerror', n_jobs=N_JOBS,
                             verbosity=0, random_state=SEMILLA)
            m.fit(dev.iloc[i_tr][F_PROD], dev.iloc[i_tr][OBJ],
                  eval_set=[(dev.iloc[i_va][F_PROD], dev.iloc[i_va][OBJ])], verbose=False)
            maes.append(mean_absolute_error(dev.iloc[i_va][OBJ], m.predict(dev.iloc[i_va][F_PROD])))
        return float(np.mean(maes))

    t0 = time.time()
    est = optuna.create_study(direction='minimize',
                              sampler=optuna.samplers.TPESampler(seed=SEMILLA),
                              pruner=optuna.pruners.MedianPruner(n_warmup_steps=5))
    est.optimize(objetivo, n_trials=60, show_progress_bar=False)
    hp = est.best_params
    print(f'  {time.time()-t0:.0f}s | mejor MAE CV={est.best_value:.4f}')

# Early stopping con 'va' para fijar el nº de árboles, y reajuste sobre train+val
m_es = XGBRegressor(**hp, n_estimators=2000, early_stopping_rounds=30,
                    objective='reg:absoluteerror', n_jobs=N_JOBS, verbosity=0,
                    random_state=SEMILLA)
m_es.fit(tr[F_PROD], tr[OBJ], eval_set=[(va[F_PROD], va[OBJ])], verbose=False)

modelo_prod = XGBRegressor(**hp, n_estimators=max(m_es.best_iteration, 10),
                           objective='reg:absoluteerror', n_jobs=N_JOBS, verbosity=0,
                           random_state=SEMILLA)
modelo_prod.fit(dev[F_PROD], dev[OBJ])

MAE_PROD = mean_absolute_error(te[OBJ], modelo_prod.predict(te[F_PROD]))
desvio = 100*abs(MAE_PROD - met_prod['MAE'])/met_prod['MAE']
print(f"\nMAE del modelo a desplegar : {MAE_PROD:.4f}")
print(f"MAE documentado (notebook 4): {met_prod['MAE']:.4f}   desvío: {desvio:.1f}%")
if desvio > 5:
    print('AVISO: el modelo reconstruido difiere >5% del documentado. Revisar antes de desplegar.')
else:
    print('Coherente con el capítulo de resultados.')

20325 filas, 23 features | dev=16260 test=4065
Hiperparámetros reutilizados del notebook 4.

MAE del modelo a desplegar : 1.5207
MAE documentado (notebook 4): 1.5207   desvío: 0.0%
Coherente con el capítulo de resultados.


---
## 2. Exportación del modelo y su contrato

Se guardan dos cosas, y ambas importan:

* **`booster.json`** — el modelo en formato nativo, estable entre versiones.
* **`modelo_produccion.json`** — el *contrato* del servicio: qué features espera, en qué
  orden, con qué horizonte y qué umbral. La API lo lee al arrancar, así que el orden de
  `features` es parte del contrato: alterarlo cambiaría el significado de cada columna.

In [3]:
(RUTA/'modelo_export').mkdir(exist_ok=True)
ruta_booster = RUTA/'modelo_export'/'booster.json'
modelo_prod.get_booster().save_model(str(ruta_booster))

config = {
    'modelo_registrado': 'viento_tarifa_h12',
    'alias': 'champion',
    'modelo_booster_path': 'modelo_export/booster.json',
    'horizonte_h': H_PROD,
    'ventana': W_PROD,
    'features': F_PROD,                     # el ORDEN forma parte del contrato
    'umbral_activacion_m_s': UMBRAL,
    'latencia_harmonie_h': LATENCIA,
    'objective': 'reg:absoluteerror',
    'hiperparametros': hp,
    'n_arboles': int(modelo_prod.n_estimators),
    'mae_test': float(MAE_PROD),
    'n_test': int(len(te)),
    'fecha_registro': pd.Timestamp.now().isoformat(),
    'mlflow_tracking_uri': 'sqlite:///mlflow.db',
}
with open(RUTA/'modelo_produccion.json', 'w') as f:
    json.dump(config, f, indent=2, ensure_ascii=False)

print(f'{ruta_booster} ({ruta_booster.stat().st_size/1024:.0f} KB)')
print(f'modelo_produccion.json: {len(F_PROD)} features, H={H_PROD}h, umbral={UMBRAL} m/s')

# Verificación: el Booster recargado debe predecir EXACTAMENTE igual que el modelo original.
import xgboost as xgb
b = xgb.Booster(); b.load_model(str(ruta_booster))
muestra = te[F_PROD].iloc[:200]
p_orig = modelo_prod.predict(muestra)
p_boost = b.predict(xgb.DMatrix(muestra, feature_names=F_PROD))
print(f'Booster recargado idéntico al original: {np.allclose(p_orig, p_boost, atol=1e-5)}')

modelo_export/booster.json (75 KB)
modelo_produccion.json: 23 features, H=12h, umbral=11.5 m/s
Booster recargado idéntico al original: True


---
## 3. Contenerización

`Dockerfile` y `docker-compose.yml` se escriben desde aquí para que el despliegue quede
documentado en un solo sitio y no se desincronice del contrato anterior.

Decisiones que merecen explicación:

* **Imagen `slim`** y solo las dependencias de servicio (sin `mlflow`, sin `optuna`): la
  imagen de inferencia no necesita nada del entrenamiento.
* **Usuario sin privilegios**: si alguien compromete el proceso, no obtiene `root` en el
  contenedor.
* **`booster.json` copiado a la imagen**, no montado: el modelo es parte de la versión
  desplegada. Montarlo permitiría cambiar el modelo sin cambiar la imagen, rompiendo la
  trazabilidad entre versión de imagen y modelo servido.
* **`healthcheck`** contra `/salud`: permite a Docker reiniciar el contenedor si el proceso
  queda vivo pero el modelo no responde.

In [4]:
DOCKERFILE = '''# Imagen de inferencia: solo lo necesario para SERVIR el modelo.
# El entrenamiento (mlflow, optuna, lightgbm, torch) queda fuera a propósito.
FROM python:3.12-slim

ENV PYTHONDONTWRITEBYTECODE=1 \\
    PYTHONUNBUFFERED=1

WORKDIR /app

COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

# El modelo y su contrato viajan DENTRO de la imagen: la versión de la imagen
# identifica unívocamente qué modelo se está sirviendo.
COPY app.py modelo_produccion.json ./
COPY modelo_export/booster.json ./modelo_export/booster.json

# Usuario sin privilegios: limita el impacto de una hipotética ejecución remota.
RUN useradd --create-home --shell /bin/bash servicio && chown -R servicio:servicio /app
USER servicio

EXPOSE 8000

HEALTHCHECK --interval=30s --timeout=5s --start-period=10s --retries=3 \\
    CMD python -c "import urllib.request; urllib.request.urlopen('http://localhost:8000/salud')"

CMD ["uvicorn", "app:app", "--host", "0.0.0.0", "--port", "8000"]
'''

COMPOSE = '''# Levanta la API de predicción, el dashboard HITL y la interfaz de MLflow.
#
#   docker compose up --build
#
#   API:        http://localhost:8000/docs
#   Dashboard:  http://localhost:8501
#   MLflow UI:  http://localhost:5050

services:
  api:
    build: .
    container_name: viento-api
    ports:
      - "8000:8000"
    restart: unless-stopped

  dashboard:
    build:
      context: .
      dockerfile: Dockerfile.dashboard
    container_name: viento-dashboard
    ports:
      - "8501:8501"
    environment:
      # Dentro de la red de compose los servicios se resuelven por su nombre.
      - API_URL=http://api:8000
    depends_on:
      - api
    volumes:
      # Solo lectura: el dashboard consulta el dataset para autocompletar features,
      # nunca lo modifica.
      - ./dataset_modelado.csv:/app/dataset_modelado.csv:ro
      - ./harmonie_crudo.csv:/app/harmonie_crudo.csv:ro
    restart: unless-stopped

  mlflow-ui:
    image: python:3.12-slim
    container_name: mlflow-ui
    working_dir: /app
    volumes:
      - ./mlflow.db:/app/mlflow.db
      - ./mlruns:/app/mlruns
    ports:
      - "5050:5000"
    command: >
      bash -c "pip install --no-cache-dir 'mlflow==3.*' &&
               mlflow ui --backend-store-uri sqlite:///mlflow.db --host 0.0.0.0 --port 5000"
    restart: unless-stopped
'''

DOCKERFILE_DASH = '''FROM python:3.12-slim

ENV PYTHONDONTWRITEBYTECODE=1 \\
    PYTHONUNBUFFERED=1

WORKDIR /app

RUN pip install --no-cache-dir \\
    "streamlit==1.*" "pandas>=2.2,<3.0" "numpy>=1.26,<3.0" "requests==2.*"

COPY streamlit_dashboard.py modelo_produccion.json ./

RUN useradd --create-home --shell /bin/bash servicio && chown -R servicio:servicio /app
USER servicio

EXPOSE 8501

CMD ["streamlit", "run", "streamlit_dashboard.py", \\
     "--server.address=0.0.0.0", "--server.port=8501", "--server.headless=true"]
'''

REQUIREMENTS = '''# Dependencias del SERVICIO de inferencia, no del entrenamiento.
# mlflow queda fuera a propósito: la API sirve el Booster nativo de XGBoost
# (ver app.py), así que no necesita el runtime de MLflow.
fastapi>=0.115,<1.0
uvicorn[standard]>=0.32,<1.0
pydantic>=2.0,<3.0
xgboost>=2.1,<4.0
pandas>=2.2,<3.0
numpy>=1.26,<3.0
'''

for nombre, contenido in [('Dockerfile', DOCKERFILE), ('Dockerfile.dashboard', DOCKERFILE_DASH),
                          ('docker-compose.yml', COMPOSE), ('requirements.txt', REQUIREMENTS)]:
    (RUTA/nombre).write_text(contenido)
    print(f'escrito {nombre} ({len(contenido.splitlines())} líneas)')

escrito Dockerfile (27 líneas)
escrito Dockerfile.dashboard (19 líneas)
escrito docker-compose.yml (48 líneas)
escrito requirements.txt (9 líneas)


---
## 4. Arranque y prueba de la API

La API se arranca como **subproceso** desde el notebook y se prueba con datos reales del
conjunto de test. Es la verificación que separa "el fichero existe" de "el servicio
funciona": se comprueba que la predicción a través de HTTP coincide con la del modelo en
memoria.

Si algo falla aquí, fallará igual en el contenedor.

In [5]:
def api_viva(url, timeout=2):
    try:
        return requests.get(f'{url}/salud', timeout=timeout).json().get('estado') == 'ok'
    except requests.exceptions.RequestException:
        return False

API_URL = f'http://localhost:{PUERTO_API}'
proc = None

if api_viva(API_URL):
    print(f'Ya hay una API respondiendo en {API_URL}; se reutiliza.')
else:
    print('Arrancando uvicorn...')
    proc = subprocess.Popen(
        [sys.executable, '-m', 'uvicorn', 'app:app', '--host', '127.0.0.1',
         '--port', str(PUERTO_API), '--log-level', 'warning'],
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for _ in range(40):                 # hasta 20 s de margen para cargar el modelo
        time.sleep(.5)
        if api_viva(API_URL):
            break
    if not api_viva(API_URL):
        salida = proc.stdout.read() if proc.stdout else ''
        proc.terminate()
        raise RuntimeError(f'La API no arrancó:\n{salida[-2000:]}')

print('Estado:', json.dumps(requests.get(f'{API_URL}/salud').json(), indent=2, ensure_ascii=False))

Ya hay una API respondiendo en http://localhost:8000; se reutiliza.
Estado: {
  "estado": "ok",
  "modelo": "viento_tarifa_h12",
  "alias": "champion",
  "horizonte_h": 12
}


In [6]:
# Prueba con filas reales del conjunto de test: la predicción vía HTTP debe coincidir
# con la del modelo en memoria. Cualquier discrepancia indicaría que la API está
# construyendo o reordenando las features de forma distinta al entrenamiento.
muestra = te[F_PROD].iloc[:25]
esperado = modelo_prod.predict(muestra)

obtenidos = []
for (ts, fila), pred_local in zip(muestra.iterrows(), esperado):
    r = requests.post(f'{API_URL}/predecir', json={
        'features': {k: float(v) for k, v in fila.items()},
        'ts_objetivo': ts.isoformat(),
    }, timeout=10)
    r.raise_for_status()
    obtenidos.append(r.json()['velocidad_predicha_m_s'])

obtenidos = np.array(obtenidos)
coincide = np.allclose(np.maximum(esperado, 0), obtenidos, atol=1e-3)
print(f'{len(obtenidos)} predicciones vía HTTP')
print(f'Coinciden con el modelo en memoria: {coincide}')
print(f'Discrepancia máxima: {np.abs(np.maximum(esperado,0)-obtenidos).max():.6f} m/s')

ejemplo = requests.post(f'{API_URL}/predecir', json={
    'features': {k: float(v) for k, v in muestra.iloc[0].items()},
    'ts_objetivo': muestra.index[0].isoformat()}).json()
print('\nRespuesta de ejemplo:')
print(json.dumps(ejemplo, indent=2, ensure_ascii=False))

# Comprobación de que la validación de entrada funciona: faltar una feature debe dar 422,
# no una predicción silenciosamente incorrecta.
incompleta = {k: float(v) for k, v in muestra.iloc[0].items()}
incompleta.pop(F_PROD[0])
r = requests.post(f'{API_URL}/predecir', json={'features': incompleta})
print(f'\nPetición con una feature de menos -> HTTP {r.status_code} '
      f'({"correcto: la rechaza" if r.status_code == 422 else "PROBLEMA: debería ser 422"})')

25 predicciones vía HTTP
Coinciden con el modelo en memoria: True
Discrepancia máxima: 0.000485 m/s

Respuesta de ejemplo:
{
  "velocidad_predicha_m_s": 14.3,
  "activar_seguridad": true,
  "umbral_m_s": 11.5,
  "horizonte_h": 12,
  "ts_objetivo": "2022-07-17T10:00:00",
  "ts_prediccion": "2026-10-02T11:52:57.413196+00:00",
  "modelo": "viento_tarifa_h12",
  "alias": "champion"
}

Petición con una feature de menos -> HTTP 422 (correcto: la rechaza)


---
## 5. Lógica Human-in-the-loop

El objetivo 6 del Anexo pide que el sistema **escale a decisión humana** cuando no sea fiable.
Una salida binaria (activar / no activar) oculta que hay predicciones al borde del umbral
donde el modelo no tiene resolución suficiente para decidir solo.

Con un MAE de referencia, la banda de incertidumbre define tres estados:

| Estado | Condición | Acción |
|---|---|---|
| **ACTIVAR** | `pred − banda ≥ umbral` | Automática: incluso en el peor caso se supera |
| **NO ACTIVAR** | `pred + banda < umbral` | Automática: ni en el mejor caso se alcanza |
| **REVISIÓN HUMANA** | el umbral cae dentro de la banda | Escalar a un operador |

La lógica se valida aquí sobre el conjunto de test —sin Streamlit— para poder cuantificar
qué fracción de horas requeriría intervención humana: un sistema que escale el 80% de los
casos no es operativo, por muy prudente que sea.

In [7]:
def clasificar(pred, umbral=UMBRAL, banda=None):
    banda = MAE_PROD if banda is None else banda
    if pred - banda >= umbral:
        return 'ACTIVAR'
    if pred + banda < umbral:
        return 'NO_ACTIVAR'
    return 'REVISION_HUMANA'

pred_te = np.maximum(modelo_prod.predict(te[F_PROD]), 0)
real_te = te[OBJ].values
estados = np.array([clasificar(p) for p in pred_te])

resumen = pd.Series(estados).value_counts().rename('horas').to_frame()
resumen['%'] = (100*resumen.horas/len(estados)).round(1)
display(resumen)

# En los casos AUTOMÁTICOS, ¿cuántas veces se equivoca? Es lo que de verdad importa:
# un error en un caso escalado lo corrige el operador; uno en automático, no.
auto = estados != 'REVISION_HUMANA'
decision = np.where(estados[auto] == 'ACTIVAR', True, False)
verdad = real_te[auto] >= UMBRAL
fallos = decision != verdad
fn = int((verdad & ~decision).sum())      # no activó y debía: el error grave
fp = int((~verdad & decision).sum())

print(f'Decisiones automáticas: {auto.sum()} de {len(estados)} ({100*auto.mean():.1f}%)')
print(f'  aciertos              : {(~fallos).sum()} ({100*(~fallos).mean():.2f}%)')
print(f'  falsos negativos      : {fn}  <- no activar cuando había que activar (crítico)')
print(f'  falsos positivos      : {fp}  <- activar sin necesidad (coste operativo, no de seguridad)')
print(f'\nHoras escaladas a revisión humana: {(~auto).sum()} ({100*(~auto).mean():.1f}%)')
print(f'  de ellas, superaban el umbral: {int((real_te[~auto] >= UMBRAL).sum())}')
print('\n(la banda usada es el MAE del modelo; ampliarla reduce falsos negativos a costa')
print(' de escalar más horas: es el compromiso que decide el operador del sistema)')

,horas,%
NO_ACTIVAR,2918,71.8
ACTIVAR,663,16.3
REVISION_HUMANA,484,11.9


Decisiones automáticas: 3581 de 4065 (88.1%)
  aciertos              : 3430 (95.78%)
  falsos negativos      : 122  <- no activar cuando había que activar (crítico)
  falsos positivos      : 29  <- activar sin necesidad (coste operativo, no de seguridad)

Horas escaladas a revisión humana: 484 (11.9%)
  de ellas, superaban el umbral: 314

(la banda usada es el MAE del modelo; ampliarla reduce falsos negativos a costa
 de escalar más horas: es el compromiso que decide el operador del sistema)


---
## 6. Dashboard Human-in-the-loop

`streamlit_dashboard.py` consume la API por HTTP en lugar de cargar el modelo directamente.
Podría parecer un rodeo, pero es la decisión correcta: el dashboard **nunca puede
desincronizarse** de lo que sirve producción, y demuestra la arquitectura real en capas
(modelo → API → cliente).

Se reescribe desde aquí para que quede alineado con el contrato de la sección 2.

---
## 6. Dashboard Human-in-the-loop

`streamlit_dashboard.py` consume la API por HTTP en lugar de cargar el modelo
directamente. Podria parecer un rodeo, pero es la decision correcta: el dashboard
**nunca puede desincronizarse** de lo que sirve produccion, y demuestra la
arquitectura real en capas (modelo -> API -> cliente).

Los tres estados de la seccion anterior son los que la interfaz presenta al operador,
que registra su decision final: esas decisiones son la materia prima del reentrenamiento.

In [8]:
DASHBOARD = '"""\nDashboard Human-in-the-loop — predicción de viento, Tarifa.\n\nConsume la API REST (app.py), no reimplementa la carga del modelo: así el\ndashboard nunca puede desincronizarse de lo que sirve producción.\n\n    streamlit run streamlit_dashboard.py\n\nRequiere la API en marcha (uvicorn o Docker):\n    curl http://localhost:8000/salud   ->   {"estado": "ok", ...}\n"""\nfrom __future__ import annotations\n\nimport json\nimport os\nfrom datetime import datetime, timezone\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\nimport requests\nimport streamlit as st\n\nRUTA = Path(__file__).parent\nAPI_URL_DEFECTO = os.environ.get("API_URL", "http://localhost:8000")\nLOG_DECISIONES = RUTA / "decisiones_hitl.csv"\nDATASET = RUTA / "dataset_modelado.csv"\nCRUDO = RUTA / "harmonie_crudo.csv"\nOBJ = "obs_vel"\n\n\n# --------------------------------------------------------------------------\n# Lógica de negocio, separada de la interfaz para poder probarla sin Streamlit.\n# --------------------------------------------------------------------------\ndef clasificar(pred: float, umbral: float, banda: float) -> str:\n    """\n    Tres estados en lugar de una decisión binaria:\n\n      ACTIVAR         -> incluso restando la banda de error se supera el umbral.\n      NO_ACTIVAR      -> incluso sumándola se queda por debajo.\n      REVISION_HUMANA -> el umbral cae dentro del margen de error del modelo:\n                         la decisión automática no es fiable, decide un operador.\n    """\n    if pred - banda >= umbral:\n        return "ACTIVAR"\n    if pred + banda < umbral:\n        return "NO_ACTIVAR"\n    return "REVISION_HUMANA"\n\n\nCOLOR = {"ACTIVAR": "#c62828", "NO_ACTIVAR": "#2e7d32", "REVISION_HUMANA": "#ef6c00"}\nTEXTO = {\n    "ACTIVAR": "ACTIVAR PROTOCOLO DE SEGURIDAD",\n    "NO_ACTIVAR": "SIN RIESGO — no se requiere acción",\n    "REVISION_HUMANA": "REVISIÓN HUMANA REQUERIDA",\n}\n\n\n@st.cache_data(ttl=600)\ndef cargar_contexto() -> pd.DataFrame | None:\n    """Dataset histórico, solo para autocompletar features de una hora concreta."""\n    if not DATASET.exists():\n        return None\n    return pd.read_csv(DATASET, index_col="ts", parse_dates=["ts"])\n\n\ndef autocompletar(ts_str: str, features: list[str], H: int, W: int) -> dict | None:\n    """\n    Reconstruye las features de una hora del histórico. Replica la lógica del\n    notebook 8: lags que terminan en t-H, más agregados de tendencia.\n    """\n    d = cargar_contexto()\n    if d is None:\n        return None\n    try:\n        ts = pd.Timestamp(ts_str)\n    except ValueError:\n        return None\n    if ts not in d.index:\n        return None\n\n    fila = d.loc[ts]\n    tramo = fila["tramo"] if "tramo" in d.columns else None\n    g = d[d.tramo == tramo].sort_index() if tramo is not None else d.sort_index()\n    if ts not in g.index:\n        return None\n    pos = g.index.get_loc(ts)\n\n    vals = {}\n    ventana = []\n    for k in range(H, H + W):\n        j = pos - k\n        if j < 0:\n            return None\n        v = float(g[OBJ].iloc[j])\n        vals[f"lag{k}"] = v\n        ventana.append(v)\n\n    for c in ("hora_sin", "hora_cos", "dia_sin", "dia_cos",\n              "H_vel", "H_dir_sin", "H_dir_cos", "H_lead"):\n        if c in g.columns:\n            vals[c] = float(fila[c])\n\n    arr = np.array(ventana)\n    vals["lag_media"] = float(arr.mean())\n    vals["lag_std"] = float(arr.std())\n    vals["lag_tend"] = float(arr[0] - arr[-1])\n\n    faltan = [f for f in features if f not in vals]\n    if faltan:\n        return None\n    return {f: vals[f] for f in features}\n\n\ndef registrar(fila: dict) -> None:\n    """Traza cada decisión: sin registro no hay auditoría ni reentrenamiento futuro."""\n    df = pd.DataFrame([fila])\n    df.to_csv(LOG_DECISIONES, mode="a", header=not LOG_DECISIONES.exists(), index=False)\n\n\n# --------------------------------------------------------------------------\n# Interfaz\n# --------------------------------------------------------------------------\ndef main() -> None:\n    st.set_page_config(page_title="Viento Tarifa — HITL", page_icon="🌬", layout="wide")\n    st.title("Predicción de viento — Tarifa")\n    st.caption("Corrección estadística de HARMONIE-AROME · sistema con supervisión humana")\n\n    with st.sidebar:\n        st.header("Configuración")\n        api_url = st.text_input("URL de la API", API_URL_DEFECTO)\n\n        try:\n            salud = requests.get(f"{api_url}/salud", timeout=3).json()\n        except requests.exceptions.RequestException as e:\n            st.error(f"API no disponible: {type(e).__name__}")\n            st.info("Arranca la API:\\\\n\\\\n`uvicorn app:app --port 8000`")\n            st.stop()\n\n        if salud.get("estado") != "ok":\n            st.error(f"La API responde pero el modelo no está cargado: {salud}")\n            st.stop()\n\n        st.success("API operativa")\n        st.metric("Horizonte", f"{salud[\'horizonte_h\']} h")\n\n        cfg = {}\n        if (RUTA / "modelo_produccion.json").exists():\n            cfg = json.loads((RUTA / "modelo_produccion.json").read_text())\n            st.metric("MAE del modelo", f"{cfg.get(\'mae_test\', float(\'nan\')):.3f} m/s")\n\n        umbral = st.number_input("Umbral de activación (m/s)", 0.0, 40.0,\n                                 float(cfg.get("umbral_activacion_m_s", 11.5)), 0.5)\n        banda = st.slider(\n            "Banda de incertidumbre (m/s)", 0.0, 5.0,\n            float(round(cfg.get("mae_test", 1.5), 2)), 0.1,\n            help="Margen alrededor del umbral dentro del cual la decisión se escala "\n                 "a un operador. Por defecto, el MAE del modelo.",\n        )\n\n    features = cfg.get("features", [])\n    H = int(salud.get("horizonte_h", 12))\n    W = int(cfg.get("ventana", 12))\n\n    tab_pred, tab_hist = st.tabs(["Predicción", "Histórico de decisiones"])\n\n    # ---------------------------------------------------------------- pestaña 1\n    with tab_pred:\n        col_in, col_out = st.columns([1, 1])\n\n        with col_in:\n            st.subheader("1. Datos de entrada")\n            modo = st.radio("Origen", ["Hora del histórico", "Entrada manual"],\n                            horizontal=True, label_visibility="collapsed")\n\n            entrada = None\n            ts_obj = None\n\n            if modo == "Hora del histórico":\n                d = cargar_contexto()\n                if d is None:\n                    st.warning("No se encuentra dataset_modelado.csv para autocompletar.")\n                else:\n                    ts_sel = st.selectbox(\n                        "Hora objetivo",\n                        options=[str(x) for x in d.index[-500:][::-1]],\n                        help="Se reconstruyen las features de esa hora igual que en entrenamiento.",\n                    )\n                    entrada = autocompletar(ts_sel, features, H, W)\n                    ts_obj = ts_sel\n                    if entrada is None:\n                        st.warning("No hay suficiente historia previa para esa hora.")\n                    else:\n                        with st.expander("Ver features reconstruidas"):\n                            st.dataframe(\n                                pd.Series(entrada, name="valor").to_frame().round(4),\n                                use_container_width=True,\n                            )\n            else:\n                st.caption("Valores principales; el resto se completa con la mediana histórica.")\n                d = cargar_contexto()\n                base_vals = {}\n                if d is not None:\n                    for f in features:\n                        if f in d.columns:\n                            base_vals[f] = float(d[f].median())\n                        elif f.startswith("lag"):\n                            base_vals[f] = float(d[OBJ].median())\n                        else:\n                            base_vals[f] = 0.0\n                else:\n                    base_vals = {f: 0.0 for f in features}\n\n                c1, c2 = st.columns(2)\n                with c1:\n                    h_vel = st.number_input("HARMONIE — velocidad (m/s)", 0.0, 40.0, 8.0, 0.5)\n                    h_dir = st.slider("HARMONIE — dirección (°)", 0, 359, 90)\n                with c2:\n                    vel_reciente = st.number_input("Velocidad observada reciente (m/s)",\n                                                   0.0, 40.0, 7.0, 0.5)\n                    hora = st.slider("Hora del día (UTC)", 0, 23, 12)\n\n                base_vals.update({\n                    "H_vel": h_vel,\n                    "H_dir_sin": float(np.sin(np.radians(h_dir))),\n                    "H_dir_cos": float(np.cos(np.radians(h_dir))),\n                    "hora_sin": float(np.sin(2 * np.pi * hora / 24)),\n                    "hora_cos": float(np.cos(2 * np.pi * hora / 24)),\n                })\n                for f in features:\n                    if f.startswith("lag") and f[3:].isdigit():\n                        base_vals[f] = vel_reciente\n                lags = [base_vals[f] for f in features if f.startswith("lag") and f[3:].isdigit()]\n                if lags:\n                    base_vals["lag_media"] = float(np.mean(lags))\n                    base_vals["lag_std"] = float(np.std(lags))\n                    base_vals["lag_tend"] = 0.0\n                entrada = {f: float(base_vals.get(f, 0.0)) for f in features}\n                ts_obj = datetime.now(timezone.utc).isoformat()\n\n        with col_out:\n            st.subheader("2. Resultado")\n            if entrada is None:\n                st.info("Selecciona o introduce los datos de entrada.")\n            elif st.button("Predecir", type="primary", use_container_width=True):\n                try:\n                    r = requests.post(f"{api_url}/predecir",\n                                      json={"features": entrada, "ts_objetivo": ts_obj},\n                                      timeout=15)\n                    r.raise_for_status()\n                    res = r.json()\n                except requests.exceptions.RequestException as e:\n                    st.error(f"Error al llamar a la API: {e}")\n                    st.stop()\n\n                pred = res["velocidad_predicha_m_s"]\n                estado = clasificar(pred, umbral, banda)\n                st.session_state["ultimo"] = {"pred": pred, "estado": estado,\n                                              "ts_obj": ts_obj, "res": res}\n\n            if "ultimo" in st.session_state:\n                u = st.session_state["ultimo"]\n                pred, estado = u["pred"], u["estado"]\n\n                st.markdown(\n                    f"<div style=\'background:{COLOR[estado]};color:white;padding:1.2rem;"\n                    f"border-radius:.5rem;text-align:center;font-size:1.15rem;"\n                    f"font-weight:600\'>{TEXTO[estado]}</div>",\n                    unsafe_allow_html=True,\n                )\n                c1, c2, c3 = st.columns(3)\n                c1.metric("Predicción", f"{pred:.2f} m/s")\n                c2.metric("Umbral", f"{umbral:.1f} m/s", delta=f"{pred - umbral:+.2f}")\n                c3.metric("Intervalo", f"±{banda:.2f}")\n                st.caption(f"Rango considerado: {max(pred - banda, 0):.2f} – {pred + banda:.2f} m/s")\n\n                if estado == "REVISION_HUMANA":\n                    st.warning(\n                        "El umbral cae dentro del margen de error del modelo. "\n                        "La decisión automática no es fiable: debe decidir un operador."\n                    )\n\n                st.subheader("3. Decisión del operador")\n                with st.form("decision"):\n                    accion = st.radio("Acción tomada",\n                                      ["Activar protocolo", "No activar", "Aplazar"],\n                                      horizontal=True)\n                    nota = st.text_input("Justificación (opcional)")\n                    if st.form_submit_button("Registrar decisión", use_container_width=True):\n                        registrar({\n                            "ts_decision": datetime.now(timezone.utc).isoformat(),\n                            "ts_objetivo": u["ts_obj"],\n                            "prediccion_m_s": pred,\n                            "umbral_m_s": umbral,\n                            "banda_m_s": banda,\n                            "estado_sistema": u["estado"],\n                            "decision_operador": accion,\n                            "coincide_con_sistema": (\n                                (accion == "Activar protocolo" and u["estado"] == "ACTIVAR")\n                                or (accion == "No activar" and u["estado"] == "NO_ACTIVAR")\n                            ),\n                            "justificacion": nota,\n                        })\n                        st.success("Decisión registrada.")\n\n    # ---------------------------------------------------------------- pestaña 2\n    with tab_hist:\n        st.subheader("Histórico de decisiones")\n        if not LOG_DECISIONES.exists():\n            st.info("Todavía no se ha registrado ninguna decisión.")\n        else:\n            log = pd.read_csv(LOG_DECISIONES)\n            c1, c2, c3 = st.columns(3)\n            c1.metric("Decisiones", len(log))\n            if "estado_sistema" in log.columns:\n                escaladas = (log.estado_sistema == "REVISION_HUMANA").sum()\n                c2.metric("Escaladas a revisión", f"{escaladas} ({100*escaladas/len(log):.0f}%)")\n            if "coincide_con_sistema" in log.columns:\n                ac = log.coincide_con_sistema.sum()\n                c3.metric("Operador de acuerdo", f"{ac} ({100*ac/len(log):.0f}%)")\n            st.dataframe(log.iloc[::-1], use_container_width=True)\n            st.download_button("Descargar CSV", log.to_csv(index=False),\n                               "decisiones_hitl.csv", "text/csv")\n            st.caption(\n                "Las discrepancias entre operador y sistema son la materia prima para "\n                "reentrenar: señalan dónde el modelo no captura el criterio experto."\n            )\n\n\nif __name__ == "__main__":\n    main()\n'

(RUTA/'streamlit_dashboard.py').write_text(DASHBOARD)
print(f'escrito streamlit_dashboard.py ({len(DASHBOARD.splitlines())} lineas)')

# Se valida que compila y que la logica HITL es correcta, sin arrancar Streamlit.
import py_compile
py_compile.compile(str(RUTA/'streamlit_dashboard.py'), doraise=True)
print('compila sin errores')

import importlib.util
spec = importlib.util.spec_from_file_location('dash_mod', RUTA/'streamlit_dashboard.py')
mod = importlib.util.module_from_spec(spec)
try:
    spec.loader.exec_module(mod)
    casos = [(14.0, 11.5, 1.5, 'ACTIVAR'), (8.0, 11.5, 1.5, 'NO_ACTIVAR'),
             (11.5, 11.5, 1.5, 'REVISION_HUMANA'), (12.5, 11.5, 1.5, 'REVISION_HUMANA')]
    ok = all(mod.clasificar(p, u, b) == esp for p, u, b, esp in casos)
    print(f'logica de clasificacion HITL correcta: {ok}')
except ImportError as e:
    print(f'(streamlit no instalado: se omite la comprobacion en vivo -- {e.name})')


escrito streamlit_dashboard.py (331 lineas)
compila sin errores


2026-10-02 13:52:57.609 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager


logica de clasificacion HITL correcta: True


---
## 7. Cierre y guía de despliegue

In [9]:
# Se detiene la API arrancada por este notebook. En producción la gestiona Docker,
# no un proceso hijo de un kernel de Jupyter.
if proc is not None:
    proc.terminate()
    try:
        proc.wait(timeout=10)
    except subprocess.TimeoutExpired:
        proc.kill()
    print('API detenida (la arrancó este notebook).')
else:
    print('La API seguía corriendo de antes: se deja como estaba.')

artefactos = ['modelo_export/booster.json', 'modelo_produccion.json', 'app.py',
              'streamlit_dashboard.py', 'Dockerfile', 'Dockerfile.dashboard',
              'docker-compose.yml', 'requirements.txt']
print()
for a in artefactos:
    p = RUTA/a
    print(f"  {'OK   ' if p.exists() else 'FALTA'} {a:34s} "
          f"{p.stat().st_size/1024:8.1f} KB" if p.exists() else f"  FALTA {a}")

print(f'\nModelo servido: XGBoost H={H_PROD}h, ventana={W_PROD}, '
      f'{len(F_PROD)} features, MAE={MAE_PROD:.4f} m/s')

La API seguía corriendo de antes: se deja como estaba.

  OK    modelo_export/booster.json             74.5 KB
  OK    modelo_produccion.json                  1.1 KB
  OK    app.py                                  7.6 KB
  OK    streamlit_dashboard.py                 13.7 KB
  OK    Dockerfile                              1.0 KB
  OK    Dockerfile.dashboard                    0.5 KB
  OK    docker-compose.yml                      1.3 KB
  OK    requirements.txt                        0.3 KB

Modelo servido: XGBoost H=12h, ventana=12, 23 features, MAE=1.5207 m/s


### Levantar el sistema completo

```bash
docker compose up --build
```

| Servicio | URL | Qué es |
|---|---|---|
| API | http://localhost:8000/docs | Documentación interactiva (OpenAPI) |
| Dashboard | http://localhost:8501 | Interfaz de decisión con supervisión humana |
| MLflow | http://localhost:5050 | Experimentos y registro de modelos |

Sin Docker, en dos terminales:

```bash
uvicorn app:app --port 8000
streamlit run streamlit_dashboard.py
```

### Ejemplo de consumo de la API

```bash
curl -X POST http://localhost:8000/predecir \
  -H 'Content-Type: application/json' \
  -d '{"features": {...}, "ts_objetivo": "2021-08-15T12:00:00"}'
```

```json
{
  "velocidad_predicha_m_s": 13.42,
  "activar_seguridad": true,
  "umbral_m_s": 11.5,
  "horizonte_h": 12
}
```

---

## Conclusiones

**Para la memoria:**

* La arquitectura en tres capas (modelo → API → cliente) evita que el dashboard pueda
  desincronizarse del modelo que sirve producción.
* Servir el `Booster` nativo en vez de `mlflow.pyfunc` resuelve dos problemas reales de
  portabilidad —rutas absolutas y compatibilidad de versiones— y elimina `mlflow` de la
  imagen de inferencia.
* La verificación de la sección 4 (predicción vía HTTP idéntica a la del modelo en memoria)
  es la prueba de ausencia de *training-serving skew*.
* Los tres estados del HITL con la banda de incertidumbre, y el porcentaje de horas que
  requieren revisión humana, cuantifican el compromiso entre automatización y prudencia:
  ampliar la banda reduce falsos negativos pero escala más casos.
* El registro de decisiones (`decisiones_hitl.csv`) cierra el ciclo: las discrepancias entre
  operador y sistema son la materia prima del reentrenamiento.

**Limitaciones a declarar:**

1. El modelo desplegado es el del horizonte de 12 h; un sistema real serviría varios
   horizontes simultáneamente.
2. La API espera las features ya construidas: el pipeline de ingesta en tiempo real (leer la
   pasada de HARMONIE, componer los lags) queda fuera del alcance de este TFM.
3. La monitorización de deriva (`evidently_monitor.py`) está preparada pero requiere datos de
   producción acumulados para ser significativa.